Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
# DBTITLE 1,Cell 1
# MAGIC
# MAGIC import trino
# MAGIC import pandas as pd
# MAGIC
# MAGIC conn = trino.dbapi.connect(
# MAGIC     host="trino-tableau-prod.dp.navi-tech.in",
# MAGIC     port=443,
# MAGIC     user="databricks-user",
# MAGIC     catalog="awsdatacatalog",
# MAGIC     schema="default",
# MAGIC     http_scheme="https",
# MAGIC )
# MAGIC
# MAGIC query = """
# MAGIC with converted_cx_ids as(
# MAGIC select date(date_parse(converted_date, '%m/%d/%Y')) as converted_date, customer_id from corp_analytics.google_sheets_hl_sales_px_audits_funnel_tracker_data
# MAGIC ),
# MAGIC
# MAGIC interaction_data as (
# MAGIC SELECT 
# MAGIC     call_id,
# MAGIC     crt_object_id,
# MAGIC     process_name,
# MAGIC     lead_name,
# MAGIC     system_disposition,
# MAGIC     hangup_details,
# MAGIC     customer_talk_time,
# MAGIC     user_name,
# MAGIC     customer_talk_time_seconds,
# MAGIC     reference_id,
# MAGIC     date(parse_datetime(partition_key, 'yyyy-MM-dd-HH')) as call_date,
# MAGIC     cci.converted_date,
# MAGIC     case when date(parse_datetime(partition_key, 'yyyy-MM-dd-HH')) = current_date - interval '1' day and cci.converted_date is null then 'd_1_nonsales_conversations'
# MAGIC          when date(parse_datetime(partition_key, 'yyyy-MM-dd-HH')) = current_date - interval '2' day and cci.converted_date is null then 'd_2_nonsales_conversations'
# MAGIC          when cci.converted_date = current_date - interval '1' day  then 'd_1_sales_conversations'
# MAGIC          when cci.converted_date = current_date - interval '2' day  then 'd_2_sales_conversations'
# MAGIC          else null
# MAGIC     end as allocation_type
# MAGIC FROM corp_analytics_non_sensitive_view.hl_ameyo_customer_level_call_details_tbl am
# MAGIC left join converted_cx_ids cci on am.reference_id = cci.customer_id
# MAGIC WHERE cast(customer_talk_time_seconds as integer) > 0
# MAGIC )
# MAGIC
# MAGIC select * from interaction_data
# MAGIC where 
# MAGIC (day_of_week(current_date) = 1 AND allocation_type IN ('d_1_nonsales_conversations', 'd_2_nonsales_conversations', 'd_1_sales_conversations', 'd_2_sales_conversations'))
# MAGIC OR
# MAGIC (day_of_week(current_date) <> 1 AND allocation_type IN ('d_1_nonsales_conversations', 'd_1_sales_conversations'))
# MAGIC """
# MAGIC
# MAGIC df = pd.read_sql(query, conn)
# MAGIC
# MAGIC if df.empty:
# MAGIC     print("Query returned no results. The DataFrame is empty.")
# MAGIC else:
# MAGIC     display(df)

In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/11vpvLqzOcRcUxOULy7B_zCoIqap01BIRJ539CIgBMSs/edit?gid=1204962317#gid=1204962317"
worksheet_name = "Master_Dump"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=1
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=1
)